In [37]:
import cv2
import re
import tkinter as tk
from ultralytics import YOLO

def output_analysis(track_results):
    if (
            track_results[0] is not None
            and track_results[0].boxes.id is not None
    ):
        boxes = track_results[0].boxes.xyxy.cpu().numpy()
        track_ids = list(track_results[0].boxes.id.int().cpu())

        for box, id in zip(boxes, track_ids):
            x1, y1, x2, y2 = map(int, box)
            print(f"ID: {id:<5} | Coords: {[x1, y1, x2, y2]}")

root = tk.Tk()
screen_width = root.winfo_screenwidth()
screen_height = root.winfo_screenheight()
root.destroy()

model_name = "yolo11n.pt"
model_dir_path = "../models/"
model_path = model_dir_path + model_name
model = YOLO(model_path)

image_name = "people_crowd.jpg"
image_dir_path = "../materials/"
image_path = image_dir_path + image_name
image = cv2.imread(image_path)

object_classes = list(map(int, re.split(r'[ ,\s\n]+',input('''Введіть коди класів об'єктів для ідентифікації через пробіл / кому:
 (0\t - люди,
 2\t - автомобілі,
 24\t - рюкзаки,
 39\t - пляшки,
 63\t - ноутбуки)
''').strip())))
confidence_coef = float(input('''Введіть коєфіцієнт впевненості моделі:
'''))

results = model.track(
    source = image,
    classes = object_classes,
    conf = confidence_coef,
    tracker = "botsort.yaml",
    persist = True
)

scale_factor = min(screen_width / image.shape[1], screen_height / image.shape[0]) / 1.5

annotated_image = results[0].plot(labels=False, conf=False)

if results[0].boxes is not None:
    boxes = results[0].boxes.xyxy.cpu().numpy()
    classes = results[0].boxes.cls.int().cpu().tolist()
    confs = results[0].boxes.conf.cpu().tolist()

    img_h, img_w = image.shape[:2]
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = max(img_w, img_h) / 2000.0
    thickness = max(int(font_scale * 2), 2)
    pad = int(font_scale * 10)

    bg_color = (255, 0, 0)
    text_color = (255, 255, 255)

    for box, cls, conf in zip(boxes, classes, confs):
        x1, y1, x2, y2 = map(int, box)
        class_name = model.names[cls]

        label = f"{class_name} {conf:.2f}"

        (w, h), baseline = cv2.getTextSize(label, font, font_scale, thickness)

        if y1 - h - pad * 2 < 0:
            bg_top_left = (x1, y1)
            bg_bottom_right = (x1 + w + pad, y1 + h + pad * 2)
            text_org = (x1 + pad // 2, y1 + h + pad)
        else:
            bg_top_left = (x1, y1 - h - pad * 2)
            bg_bottom_right = (x1 + w + pad, y1)
            text_org = (x1 + pad // 2, y1 - pad)

        cv2.rectangle(annotated_image, bg_top_left, bg_bottom_right, bg_color, -1)
        cv2.putText(
            img=annotated_image,
            text=label,
            org=text_org,
            fontFace=font,
            fontScale=font_scale,
            color=text_color,
            thickness=thickness
        )


resized_image = cv2.resize(annotated_image, None, fx=scale_factor, fy=scale_factor, interpolation=cv2.INTER_AREA)

cv2.namedWindow("result", cv2.WINDOW_AUTOSIZE)

cv2.imwrite("../materials/result_" + image_name, resized_image)
cv2.imshow("result", resized_image)

output_analysis(results)

cv2.waitKey(0)
cv2.destroyAllWindows()


0: 640x448 15 persons, 29.5ms
Speed: 1.8ms preprocess, 29.5ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 448)
ID: 1     | Coords: [1533, 5568, 1743, 5904]
ID: 2     | Coords: [947, 5422, 1232, 5902]
ID: 3     | Coords: [751, 4814, 979, 5508]
ID: 4     | Coords: [2761, 4596, 2971, 5270]
ID: 5     | Coords: [406, 4750, 667, 5492]
ID: 6     | Coords: [3248, 3409, 3451, 3939]
ID: 7     | Coords: [2394, 4739, 2622, 5395]
ID: 8     | Coords: [2772, 4202, 2975, 4723]
ID: 9     | Coords: [1820, 469, 1968, 935]
ID: 10    | Coords: [3055, 2345, 3224, 2918]
ID: 11    | Coords: [3613, 4881, 3860, 5580]
ID: 12    | Coords: [1947, 2568, 2150, 3169]
ID: 13    | Coords: [1932, 3685, 2126, 4197]
ID: 14    | Coords: [1665, 2024, 1858, 2572]
ID: 15    | Coords: [2468, 2083, 2637, 2570]
